In [1]:
from pyspark.sql import SparkSession
spark=(
    SparkSession.builder
    .appName("RDD Transformation")
    .master("local[*]")
    .getOrCreate()
)
sc=spark.sparkContext
print(sc.uiWebUrl)

sc

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/21 20:40:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


http://macbookair.lan:4040


<SparkContext master=local[*] appName=RDD Transformation>

# Narrow Transformation 

In [ ]:
# map() - This is an RDD transformation that applies a function to each element of an RDD and returns exactly one output element for each input element.

# map() -> 1 input -> 1 output 

#new_rdd=rdd.map(function)

numbers=[10,20,30,40,50]
rdd=sc.parallelize(numbers)
result=rdd.map(lambda x:x*2)
result.collect()


In [ ]:
def multiple_by_10(x):
    return x*10

In [ ]:
result1=rdd.map(multiple_by_10)
result1.collect()

In [ ]:
# Important 

rdd=sc.parallelize([
    "Vijay",
    "Vamshi",
    "Darshan",
    "Ashad"
])

In [ ]:
result=rdd.map(lambda name:(name,len(name)))

In [ ]:
result.collect()

In [ ]:
row_data=[
    "T001,C101,5000,India",
    "T002,C102,6000,Usa",
    "T003,C103,59000,uk",
    "T004,C104,5400,Usa",
    "T005,C105,50400,India"

]

In [ ]:
rdd_row_data=sc.parallelize(row_data)
result_row_data=rdd_row_data.map(lambda x:x.split(","))
result_row_data.collect()

In [ ]:
def parse_transformation(record):
    txn_id,customer_id,amount,country=record.split(",")
    return(
        txn_id,
        customer_id,
        float(amount),
        country.upper()
    )

In [ ]:
transection_rdd=rdd_row_data.map(parse_transformation)


In [ ]:
transection_rdd.collect()

### Map() Usecases 

- Parsing records 
- Type Conversion 
- Extacting Values 
- Adding calculated fields 
- Converting one struture into another 
- Creating key-value RDD
- Applying Bsuiness Logic 



In [ ]:
transections=[
    ('T001', 'C101', 5000.0, 'INDIA'),
    ('T002', 'C102', 6000.0, 'USA'),
    ('T003', 'C103', 59000.0, 'UK'),
    ('T004', 'C104', 5400.0, 'USA'),
    ('T005', 'C105', 50400.0, 'INDIA')]
rdd=sc.parallelize(transections)

In [ ]:
customer_amount=rdd.map(lambda x:(x[1],x[2]))
customer_amount.collect()

In [ ]:
rdd=sc.parallelize([1,2,3])
result=rdd.map(lambda x:[x,x*10])
result.collect()

In [ ]:
# Map() and Task Execution 

rdd=sc.parallelize(range(1_000_000),4)
result=rdd.map(lambda x:x*2)
result.collect()

In [ ]:
# Multiple Narrow Transformation 

rdd=sc.parallelize(range(100),4)
rdd2=rdd.map(lambda x:x*2)
rdd3=rdd2.map(lambda x:x*10)
rdd4=rdd3.filter(lambda x:x>50)
rdd4.collect()

## Questions map() Transformation 

Q1 — Employee Salary Hike

employees = [
    ("E101", "An", "Data Engineer", 80000),
    ("E102", "Rahul", "Developer", 60000),
    ("E103", "Priya", "Data Engineer", 90000),
    ("E104", "Amit", "Tester", 50000)
]

rdd = sc.parallelize(employees)


- Using only map(), apply these hikes:

Data Engineer → 15%
Developer     → 10%
Tester        → 5%

(employee_id, name, role, old_salary, new_salary)


Q2 — Transaction Risk Classification

transactions = [
    ("T001", "C101", 5000, "India"),
    ("T002", "C102", 18000, "USA"),
    ("T003", "C103", 75000, "India"),
    ("T004", "C104", 25000, "UK"),
    ("T005", "C105", 120000, "USA")
]

rdd = sc.parallelize(transactions)

Rules: 

amount < 10,000              → LOW
10,000 <= amount < 50,000    → MEDIUM
amount >= 50,000             → HIGH

Expected Structure:

(transaction_id, customer_id, amount, country, risk)

Q3 — Parse Raw CSV Records 

raw_data = [
    "E101,Anuj,Data Engineer,85000,India",
    "E102,Rahul,Developer,65000,USA",
    "E103,Priya,Manager,120000,India"
]

rdd = sc.parallelize(raw_data)


- Using map(), convert each string into a tuple and convert salary to int.
- Also uppercase the country.


Q4 — E-commerce Order Calculation

orders = [
    ("O101", "Laptop", 2, 50000),
    ("O102", "Mouse", 5, 1000),
    ("O103", "Keyboard", 3, 2000),
    ("O104", "Monitor", 2, 15000)
]

rdd = sc.parallelize(orders)

Fields:

(order_id, product, quantity, unit_price)

Calculate:

total_amount = quantity × unit_price
GST          = total_amount × 18%
final_amount = total_amount + GST

Expected Structure:

(order_id, product, total_amount, gst, final_amount)

-- Example :

("O101", "Laptop", 100000, 18000, 118000)
("O102", "Mouse",    5000,   900,   5900)


Q5 — Create a Pair RDD for Future Aggregation

sales = [
    ("S001", "C101", "Laptop", 50000),
    ("S002", "C102", "Mobile", 30000),
    ("S003", "C101", "Keyboard", 5000),
    ("S004", "C103", "Monitor", 20000),
    ("S005", "C102", "Mouse", 2000)
]

rdd = sc.parallelize(sales)

Transform this into a Pair RDD where:

Key   = Customer ID
Value = (Product, Amount)


Expected:

("C101", ("Laptop", 50000))
("C102", ("Mobile", 30000))
("C101", ("Keyboard", 5000))
("C103", ("Monitor", 20000))
("C102", ("Mouse", 2000))

Q6 — Data Quality Flagging

customers = [
    ("C101", "Anuj", 32, "anuj@gmail.com"),
    ("C102", "", 28, "rahul@gmail.com"),
    ("C103", "Priya", -5, "priya@gmail.com"),
    ("C104", "Amit", 45, ""),
    ("C105", "Neha", 25, "neha@gmail.com")
]

rdd = sc.parallelize(customers)


Rules:

name is empty  → INVALID_NAME

age <= 0       → INVALID_AGE

email is empty → INVALID_EMAIL

otherwise      → VALID


Expected Struccure:

(customer_id, name, age, email, status)

Expected:

("C101", "Anuj",  32, "anuj@gmail.com",  "VALID")
("C102", "",      28, "rahul@gmail.com", "INVALID_NAME")
("C103", "Priya", -5, "priya@gmail.com", "INVALID_AGE")
("C104", "Amit",  45, "",                 "INVALID_EMAIL")
("C105", "Neha",  25, "neha@gmail.com",  "VALID")


Q7 — Multiple Business Rules

transactions = [
    ("T001", "C101", 5000, "India", "UPI"),
    ("T002", "C102", 60000, "India", "CARD"),
    ("T003", "C103", 120000, "USA", "CARD"),
    ("T004", "C104", 8000, "UK", "CASH"),
    ("T005", "C105", 90000, "India", "UPI")
]

rdd = sc.parallelize(transactions)

Create :

(transaction_id,
 customer_id,
 amount,
 country,
 payment_method,
 risk_score,
 risk_category)

 Risk Score:

 amount >= 100000        → +3
amount >= 50000         → +2
otherwise               → +1

country != "India"      → +2

payment_method == CARD  → +1

Risk category:

score >= 5 → HIGH
score >= 3 → MEDIUM
otherwise  → LOW


Example:

T003
Amount 120000 → +3
USA           → +2
CARD          → +1
                 --
Score            6

Risk = HIGH


Expected T003: ("T003", "C103", 120000, "USA", "CARD", 6, "HIGH")






In [ ]:
# ## Questions map() Transformation 

# Q1 — Employee Salary Hike

# employees = [
#     ("E101", "An", "Data Engineer", 80000),
#     ("E102", "Rahul", "Developer", 60000),
#     ("E103", "Priya", "Data Engineer", 90000),
#     ("E104", "Amit", "Tester", 50000)
# ]

# rdd = sc.parallelize(employees)


# - Using only map(), apply these hikes:

# Data Engineer → 15%
# Developer     → 10%
# Tester        → 5%

# (employee_id, name, role, old_salary, new_salary)


# Q2 — Transaction Risk Classification

# transactions = [
#     ("T001", "C101", 5000, "India"),
#     ("T002", "C102", 18000, "USA"),
#     ("T003", "C103", 75000, "India"),
#     ("T004", "C104", 25000, "UK"),
#     ("T005", "C105", 120000, "USA")
# ]

# rdd = sc.parallelize(transactions)

# Rules: 

# amount < 10,000              → LOW
# 10,000 <= amount < 50,000    → MEDIUM
# amount >= 50,000             → HIGH

# Expected Structure:

# (transaction_id, customer_id, amount, country, risk)

# Q3 — Parse Raw CSV Records 

# raw_data = [
#     "E101,Anuj,Data Engineer,85000,India",
#     "E102,Rahul,Developer,65000,USA",
#     "E103,Priya,Manager,120000,India"
# ]

# rdd = sc.parallelize(raw_data)


# - Using map(), convert each string into a tuple and convert salary to int.
# - Also uppercase the country.


# Q4 — E-commerce Order Calculation

# orders = [
#     ("O101", "Laptop", 2, 50000),
#     ("O102", "Mouse", 5, 1000),
#     ("O103", "Keyboard", 3, 2000),
#     ("O104", "Monitor", 2, 15000)
# ]

# rdd = sc.parallelize(orders)

# Fields:

# (order_id, product, quantity, unit_price)

# Calculate:

# total_amount = quantity × unit_price
# GST          = total_amount × 18%
# final_amount = total_amount + GST

# Expected Structure:

# (order_id, product, total_amount, gst, final_amount)

# -- Example :

# ("O101", "Laptop", 100000, 18000, 118000)
# ("O102", "Mouse",    5000,   900,   5900)


# Q5 — Create a Pair RDD for Future Aggregation

# sales = [
#     ("S001", "C101", "Laptop", 50000),
#     ("S002", "C102", "Mobile", 30000),
#     ("S003", "C101", "Keyboard", 5000),
#     ("S004", "C103", "Monitor", 20000),
#     ("S005", "C102", "Mouse", 2000)
# ]

# rdd = sc.parallelize(sales)

# Transform this into a Pair RDD where:

# Key   = Customer ID
# Value = (Product, Amount)


# Expected:

# ("C101", ("Laptop", 50000))
# ("C102", ("Mobile", 30000))
# ("C101", ("Keyboard", 5000))
# ("C103", ("Monitor", 20000))
# ("C102", ("Mouse", 2000))

# Q6 — Data Quality Flagging

# customers = [
#     ("C101", "Anuj", 32, "anuj@gmail.com"),
#     ("C102", "", 28, "rahul@gmail.com"),
#     ("C103", "Priya", -5, "priya@gmail.com"),
#     ("C104", "Amit", 45, ""),
#     ("C105", "Neha", 25, "neha@gmail.com")
# ]

# rdd = sc.parallelize(customers)


# Rules:

# name is empty  → INVALID_NAME

# age <= 0       → INVALID_AGE

# email is empty → INVALID_EMAIL

# otherwise      → VALID


# Expected Struccure:

# (customer_id, name, age, email, status)

# Expected:

# ("C101", "Anuj",  32, "anuj@gmail.com",  "VALID")
# ("C102", "",      28, "rahul@gmail.com", "INVALID_NAME")
# ("C103", "Priya", -5, "priya@gmail.com", "INVALID_AGE")
# ("C104", "Amit",  45, "",                 "INVALID_EMAIL")
# ("C105", "Neha",  25, "neha@gmail.com",  "VALID")


# Q7 — Multiple Business Rules

# transactions = [
#     ("T001", "C101", 5000, "India", "UPI"),
#     ("T002", "C102", 60000, "India", "CARD"),
#     ("T003", "C103", 120000, "USA", "CARD"),
#     ("T004", "C104", 8000, "UK", "CASH"),
#     ("T005", "C105", 90000, "India", "UPI")
# ]

# rdd = sc.parallelize(transactions)

# Create :

# (transaction_id,
#  customer_id,
#  amount,
#  country,
#  payment_method,
#  risk_score,
#  risk_category)

#  Risk Score:

#  amount >= 100000        → +3
# amount >= 50000         → +2
# otherwise               → +1

# country != "India"      → +2

# payment_method == CARD  → +1

# Risk category:

# score >= 5 → HIGH
# score >= 3 → MEDIUM
# otherwise  → LOW


# Example:

# T003
# Amount 120000 → +3
# USA           → +2
# CARD          → +1
#                  --
# Score            6

# Risk = HIGH


# Expected T003: ("T003", "C103", 120000, "USA", "CARD", 6, "HIGH")




# # 

In [ ]:
# flatmap() : Applies a function to every element in RDD but one input element can produce zero,one or multiple output element.

# map() - I input -> 1 Output 

# flatmap() - 1 Input -> 0,1, or many outputs 

# flatmap() -> MAP() -> Transform each element   Flat-> Flatten the results 



In [ ]:
rdd=sc.parallelize([
    "Apache Spark",
    "Data Engineering",
    "Big Data"
])

In [ ]:
result=rdd.map(lambda x:x.split(" "))
result.collect()

In [ ]:
result=rdd.flatMap(lambda x:x.split(" "))
result.collect()

In [ ]:
rdd=sc.parallelize([
    "Spark",
    "",
    "Python"
])

In [ ]:
result=rdd.flatMap(lambda x:[x] if x != "" else [])

In [ ]:
result.collect()

In [ ]:
# Does flatMap() changes the Number of partition - No 

rdd=sc.parallelize([
    "Apache Spark",
    "Data Engineering",
    "Big Data"
],3)

rdd.getNumPartitions()




In [ ]:
result=rdd.flatMap(lambda x:x.split(" "))
result.getNumPartitions()

In [ ]:
result.collect()

In [ ]:
logs=[
    "ERROR database connection Failed",
    "INFO application started",
    "ERROR payment service timeout"
]

rdd=sc.parallelize(logs)

In [ ]:
words=rdd.flatMap(lambda line:line.split())

In [ ]:
words.collect()

In [ ]:
orders=[
    ("0101",["Laptop","Mouse"]),
    ("0102",["Keyboard"]),
    ("0103",["Monitor","Mouse","Keyboard"])
]

rdd=sc.parallelize(orders)

In [ ]:
result=rdd.flatMap(
    lambda x:[(x[0],product) for product in x[1]]
)

result.collect()

In [ ]:
customer=[
    ("C101",[5000,3000,7000]),
    ("C102",[10000,20000]),
    ("C103",[]),

]
rdd=sc.parallelize(customer)

In [ ]:
result=rdd.flatMap(
    lambda x:[(x[0],amount) for amount in x[1]]
)
result.collect()

In [ ]:
x=("C101",[])
output=[]
for amount in x[1]:
    output.append((x[0],amount))

In [ ]:
output

In [ ]:
# filter() : Returns only those RDD elements that staisfy a condition 

# new_rdd=rdd.filter(function)

# new_rdd=rdd.filter(lambda x: condition)

rdd=sc.parallelize([10,15,20,25,30])
result=rdd.filter(lambda x:x>20)
result.collect()



In [ ]:
rdd=sc.parallelize([1,2,3,4,5,6,7,7,8,8,8,78,6,6,56,5,4,4,3,3,3])

In [ ]:
result=rdd.filter(lambda x:x%2==0)
result.collect()

In [ ]:
logs=sc.parallelize([
    "INFO Application started",
    "ERROR Database connection failed",
    "WARN memory useage high",
    "ERROR payment service timeout",
    "INFO Application completed"
])



In [ ]:
errors=logs.filter(
    lambda line : "ERROR" in line
)

errors.collect()

# ── WIDE TRANSFORMATIONS
SET / DATASET
distinct()
intersection()
subtract()

In [ ]:
# distinct() : Removes duplicate elemnts from an RDD

# distinct(numPartitions) : The desired number of partition for the result \

-   rdd.distinct(numPartitions=10)

In [ ]:
rdd=sc.parallelize([
    10,20,10,30,20,40
])
result=rdd.distinct()
result.collect()

In [ ]:
events=sc.parallelize([
    ("U101","Product_1"),
    ("U102","Product_2"),
    ("U101","Product_1"),
    ("U103","Product_3"),
    ("U102","Product_2"),
    ("U102","Product_2"),
    ("U102","Product_2"),
    ("U101","Product_1"),
])

# Find unique (user,product) interatction 

unique_events=events.distinct()
unique_events.collect()


1 TB -> 500 Partitions 

rdd.distinct()

Spark may need significant:

- Network I/O 
- Disk I/O
- Serialization 
- Shuffle Read/write 
- memory 

### Intersection - Finds the elements thta are present in both RDDs

RDD_1 and RDD_2 --> Common element 

rdd1.intersection(rdd2)





subtract() : - Returns elements that are present in the RDD but not in the second RDD

rdd.subtract(rdd2)

A-B - In A but Not B 

### Aggregation : 

- groupby()
- reduceBykey()
- groupBykey()
- aggregateByKey()
- combineByKey()
- foldByKey() 




### groupby()

- Create a Key using a function -> Bring records having the same key togather

- GroupBy() groups RDD elements based on a key that we calculate using a function 

- groupBy(function) -> Function return Value -> Become the group Key 





In [ ]:
rdd=sc.parallelize([
    10,15,20,25,30,35
])

# Even No togather and Odd no togather 



In [ ]:
result=rdd.groupBy(
    lambda x:"EVEN" if x % 2==0 else "ODD"
)
#result.collect()
for key,values in result.collect():
    print(key,list(values))

In [ ]:
# Example 1- groupBy()

transactions = sc.parallelize([
    ("T001", "C101", 5000, "INDIA"),
    ("T002", "C102", 18000, "USA"),
    ("T003", "C103", 75000, "INDIA"),
    ("T004", "C104", 120000, "UK"),
    ("T005", "C105", 35000, "INDIA"),
    ("T006", "C106", 95000, "USA"),
    ("T007", "C107", 8000, "UK")
], 3)

# Requirment 

# - Group Transection 
    # - LOW -> amount < 10000
    # - MEDIUM -> 10000 <= amount < 50000
    # - HIGH 5000 <= amount < 100000
    # - CRITICAL - amount >=100000

In [ ]:
def get_amount_category(record):
    amount=record[2]
    if amount >=100000:
        return "CRITICAL"
    elif amount >=50000:
        return "HIGH"
    elif amount >=10000:
        return "MEDIUM"
    else:
        return "LOW"


In [ ]:
result=transactions.groupBy(get_amount_category)

In [ ]:
for category,records in result.collect():
    print("\n",category)
    for record in records:
        print(record)

In [ ]:
# Example :2 
logs = sc.parallelize([
    ("2026-08-18 10:01", "payment", "ERROR", "Database connection timeout"),
    ("2026-08-18 10:02", "order", "ERROR", "API connection timeout"),
    ("2026-08-18 10:03", "payment", "ERROR", "Invalid authentication token"),
    ("2026-08-18 10:04", "customer", "ERROR", "Database connection refused"),
    ("2026-08-18 10:05", "order", "ERROR", "Out of memory"),
    ("2026-08-18 10:06", "payment", "ERROR", "Authentication failed")
], 3)

# Requirments :

# message contains "Database"
# -> database_Error 

# message contains "timeout"
# -> TIMEOUT_ERROR

# Message contains "authentication"
# - AUTH_ERROR

# Message Contains "memory"
# -> "MEMORY_ERROR"

# otherwise 
# -> Other_ERROR 



In [ ]:
def classify_error(record):
    message=record[3].lower()

    if "database" in message:
        return "DATABASE_ERROR"
    elif "timeout" in message:
        return "TIMEOUT_ERROR"
    elif "authentication" in message:
        return "AUTH_ERROR"
    elif "memory" in message:
        return "MEMORY_ERROR"
    else:
        return "OTHER_ERROR"
    

In [ ]:
result=logs.groupBy(classify_error)


In [ ]:
for category,records in result.collect():
    print("\n",category)
    for record in records:
        print(record)

### Example : 3 



### - reduceBykey()
- Works on pair RDD (key,value)

- It combines all values belonging to the same key using a function 



In [ ]:
#Example -1 ReduceByKey()

sales =sc.parallelize([
    ("INDIA",1000),
    ("USA",7000),
    ("INDIA",5000),
    ("US",3000),
    ("UK",9000),
    ("USA",6000),
    ("USA",5000),
    ("INDIA",3000),
    ("UK",2000),
])

# Requirment:

# Calculate total sales for each country

In [ ]:
result=sales.reduceByKey(lambda x,y:x+y)

In [ ]:
result.collect()

In [ ]:
# example 2: reduceByKey()

transactions = sc.parallelize([
    ("T001", "C101", 5000, "INDIA"),
    ("T002", "C102", 18000, "USA"),
    ("T003", "C103", 75000, "INDIA"),
    ("T004", "C104", 120000, "UK"),
    ("T005", "C105", 35000, "INDIA"),
    ("T006", "C106", 95000, "USA"),
    ("T007", "C107", 8000, "UK")
], 3)

# Requirment: Calculate total transection amount by country

In [ ]:
pair_rdd=transactions.map(
    lambda x:(x[3],x[2])
)

In [ ]:
def add_values(x,y):
    return x+y

In [ ]:
result=pair_rdd.reduceByKey(add_values)

result.collect()

# ============================================================
# PYSPARK RDD PRACTICE
#
## Topics :
## distinct()
## union()
## intersection()
## subtract()
## filter()
## groupBy()
## reduceByKey()
## map()
## flatMap()
#
##Goal:
## Read the requirement and decide yourself which
## transformations are needed.
## ============================================================


## ============================================================
# QUESTION 1
# Consolidate Failed Customers Across Two Days
# Level: MEDIUM
# ============================================================

day1 = sc.parallelize([
    ("T001", "C101", 1200, "SUCCESS"),
    ("T002", "C102", 2500, "FAILED"),
    ("T003", "C103", 1800, "FAILED"),
    ("T004", "C104", 3200, "SUCCESS"),
    ("T005", "C105", 900,  "SUCCESS")
])

day2 = sc.parallelize([
    ("T006", "C102", 2100, "FAILED"),
    ("T007", "C104", 4500, "FAILED"),
    ("T008", "C106", 5100, "SUCCESS"),
    ("T009", "C103", 1400, "FAILED"),
    ("T010", "C107", 3000, "FAILED")
])

# REQUIREMENT:
#
# 1. Combine both days of transaction data.
# 2. Keep only FAILED transactions.
# 3. Extract customer IDs.
# 4. A customer may fail multiple times.
# 5. Return every failed customer only once.
#
# Expected output:
#
# C102
# C103
# C104
# C107


# ============================================================
# QUESTION 2
# Successful Customers Present in Both Months
# Level: MEDIUM
# ============================================================

july = sc.parallelize([
    ("T001", "C101", 1200, "SUCCESS"),
    ("T002", "C102", 3000, "FAILED"),
    ("T003", "C103", 2500, "SUCCESS"),
    ("T004", "C104", 1800, "SUCCESS"),
    ("T005", "C101", 900,  "SUCCESS"),
    ("T006", "C105", 5000, "SUCCESS")
])

august = sc.parallelize([
    ("T101", "C101", 2000, "SUCCESS"),
    ("T102", "C103", 1500, "FAILED"),
    ("T103", "C104", 3500, "SUCCESS"),
    ("T104", "C106", 4200, "SUCCESS"),
    ("T105", "C105", 2200, "SUCCESS"),
    ("T106", "C107", 9000, "SUCCESS")
])

# REQUIREMENT:
#
# Find customers who had at least one SUCCESSFUL transaction
# in BOTH July and August.
#
# A customer may have multiple transactions in a month.
#
# Expected:
#
# C101
# C104
# C105


# ============================================================
# QUESTION 3
# Customers Who Stopped Transacting
# Level: MEDIUM
# ============================================================

july_transactions = sc.parallelize([
    ("T001", "C101", 1000),
    ("T002", "C102", 2000),
    ("T003", "C103", 3000),
    ("T004", "C101", 1500),
    ("T005", "C104", 4000),
    ("T006", "C105", 5000)
])

august_transactions = sc.parallelize([
    ("T101", "C101", 2500),
    ("T102", "C103", 3500),
    ("T103", "C106", 6000),
    ("T104", "C103", 1200),
    ("T105", "C105", 1700)
])

# REQUIREMENT:
#
# Find customers who were active in July
# but had NO transaction at all in August.
#
# Expected:
#
# C102
# C104


# ============================================================
# QUESTION 4
# High-Value International Spend Per Customer
# Level: MEDIUM-HARD
# ============================================================

transactions = sc.parallelize([
    ("T001", "C101", 12000, "INDIA",   "SUCCESS"),
    ("T002", "C101", 45000, "USA",     "SUCCESS"),
    ("T003", "C102", 28000, "UK",      "SUCCESS"),
    ("T004", "C102", 32000, "USA",     "FAILED"),
    ("T005", "C101", 18000, "UAE",     "SUCCESS"),
    ("T006", "C103", 70000, "INDIA",   "SUCCESS"),
    ("T007", "C103", 25000, "GERMANY", "SUCCESS"),
    ("T008", "C104", 9000,  "USA",     "SUCCESS"),
    ("T009", "C104", 27000, "UK",      "SUCCESS"),
    ("T010", "C105", 56000, "UAE",     "FAILED")
])

# REQUIREMENT:
#
# Calculate total international spend for every customer.
#
# Conditions:
#
# status must be SUCCESS
# country must NOT be INDIA
# amount must be >= 10000
#
# Expected:
#
# C101 -> 63000
# C102 -> 28000
# C103 -> 25000
# C104 -> 27000


# ============================================================
# QUESTION 5
# Customers Crossing Spending Threshold
# Level: MEDIUM-HARD
# ============================================================

transactions = sc.parallelize([
    ("C101", 12000),
    ("C102", 8000),
    ("C101", 18000),
    ("C103", 25000),
    ("C102", 7000),
    ("C101", 22000),
    ("C103", 9000),
    ("C104", 45000),
    ("C104", 10000),
    ("C105", 15000)
])

# REQUIREMENT:
#
# Calculate total spend per customer.
#
# Then return only customers whose TOTAL spend
# is >= 40000.
#
# Expected:
#
# C101 -> 52000
# C104 -> 55000


# ============================================================
# QUESTION 6
# Source vs Target Reconciliation
# Level: HARD
# ============================================================

source = sc.parallelize([
    ("T001", "C101", 1000, "SUCCESS"),
    ("T002", "C102", 2000, "SUCCESS"),
    ("T003", "C103", 3000, "FAILED"),
    ("T004", "C104", 4000, "SUCCESS"),
    ("T005", "C105", 5000, "SUCCESS"),
    ("T006", "C106", 6000, "SUCCESS"),
    ("T007", "C107", 7000, "FAILED")
])

target = sc.parallelize([
    ("T001", "C101", 1000),
    ("T002", "C102", 2000),
    ("T004", "C104", 4000),
    ("T008", "C108", 8000)
])

# REQUIREMENT:
#
# Target should contain ONLY successful source transactions.
#
# Find three outputs:
#
# 1. MATCHED transaction IDs
#    Successful source IDs that exist in target.
#
# 2. MISSING transaction IDs
#    Successful source IDs that do NOT exist in target.
#
# 3. EXTRA transaction IDs
#    IDs present in target but not present in successful source.
#
# Expected:
#
# MATCHED
# T001
# T002
# T004
#
# MISSING
# T005
# T006
#
# EXTRA
# T008


# ============================================================
# QUESTION 7
# Unique Error Applications Across Two Servers
# Level: HARD
# ============================================================

server1_logs = sc.parallelize([
    ("2026-08-18 10:01", "PAYMENT", "ERROR",   "Timeout"),
    ("2026-08-18 10:02", "AUTH",    "INFO",    "Login"),
    ("2026-08-18 10:03", "ORDER",   "ERROR",   "Database unavailable"),
    ("2026-08-18 10:04", "PAYMENT", "ERROR",   "HTTP 500"),
    ("2026-08-18 10:05", "SEARCH",  "WARNING", "Slow response")
])

server2_logs = sc.parallelize([
    ("2026-08-18 10:06", "AUTH",    "ERROR", "Invalid token"),
    ("2026-08-18 10:07", "PAYMENT", "ERROR", "Connection refused"),
    ("2026-08-18 10:08", "ORDER",   "INFO",  "Order created"),
    ("2026-08-18 10:09", "PROFILE", "ERROR", "Service unavailable")
])

# REQUIREMENT:
#
# Combine logs from both servers.
#
# Keep ERROR logs only.
#
# Return application names that generated errors.
#
# Application name should appear only once.
#
# Expected:
#
# PAYMENT
# ORDER
# AUTH
# PROFILE


# ============================================================
# QUESTION 8
# Count Errors Per Application Across Multiple Servers
# Level: HARD
# ============================================================

server1_logs = sc.parallelize([
    ("PAYMENT", "ERROR"),
    ("AUTH",    "SUCCESS"),
    ("ORDER",   "ERROR"),
    ("PAYMENT", "ERROR"),
    ("SEARCH",  "SUCCESS")
])

server2_logs = sc.parallelize([
    ("AUTH",    "ERROR"),
    ("PAYMENT", "ERROR"),
    ("ORDER",   "SUCCESS"),
    ("PROFILE", "ERROR"),
    ("PAYMENT", "SUCCESS")
])

server3_logs = sc.parallelize([
    ("PAYMENT", "ERROR"),
    ("AUTH",    "ERROR"),
    ("ORDER",   "ERROR"),
    ("PROFILE", "SUCCESS")
])

# REQUIREMENT:
#
# Combine logs from all three servers.
#
# Keep ERROR records only.
#
# Calculate number of errors per application.
#
# Expected:
#
# PAYMENT -> 4
# AUTH    -> 2
# ORDER   -> 2
# PROFILE -> 1


# ============================================================
# QUESTION 9
# Group Transactions Into Risk Categories
# Level: HARD
# ============================================================

transactions = sc.parallelize([
    ("T001", "C101", 5000,   "INDIA"),
    ("T002", "C102", 25000,  "USA"),
    ("T003", "C103", 75000,  "UK"),
    ("T004", "C104", 110000, "UAE"),
    ("T005", "C105", 45000,  "INDIA"),
    ("T006", "C106", 95000,  "GERMANY"),
    ("T007", "C107", 8000,   "USA"),
    ("T008", "C108", 55000,  "INDIA"),
    ("T009", "C109", 150000, "USA")
])

# REQUIREMENT:
#
# Ignore transactions from INDIA.
#
# For remaining transactions, group the COMPLETE records
# into risk categories based on amount.
#
# amount < 10000
#     LOW
#
# amount >= 10000 and amount < 50000
#     MEDIUM
#
# amount >= 50000 and amount < 100000
#     HIGH
#
# amount >= 100000
#     CRITICAL
#
# Expected conceptual output:
#
# LOW
#     T007
#
# MEDIUM
#     T002
#
# HIGH
#     T003
#     T006
#
# CRITICAL
#     T004
#     T009


# ============================================================
# QUESTION 10
# Customer Transaction Summary
# Level: HARD
# ============================================================

transactions = sc.parallelize([
    ("T001", "C101", 10000, "SUCCESS"),
    ("T002", "C102", 8000,  "SUCCESS"),
    ("T003", "C101", 15000, "FAILED"),
    ("T004", "C103", 25000, "SUCCESS"),
    ("T005", "C101", 22000, "SUCCESS"),
    ("T006", "C102", 7000,  "SUCCESS"),
    ("T007", "C103", 9000,  "FAILED"),
    ("T008", "C104", 45000, "SUCCESS"),
    ("T009", "C101", 13000, "SUCCESS"),
    ("T010", "C104", 5000,  "SUCCESS")
])

# REQUIREMENT:
#
# Consider only SUCCESS transactions.
#
# For every customer calculate:
#
# 1. Total successful transaction amount
# 2. Number of successful transactions
#
# Expected:
#
# C101 -> (45000, 3)
# C102 -> (15000, 2)
# C103 -> (25000, 1)
# C104 -> (50000, 2)


# ============================================================
# QUESTION 11
# Active Fraud-Watch Customers With High Spend
# Level: HARD
# ============================================================

fraud_watchlist = sc.parallelize([
    "C101",
    "C103",
    "C105",
    "C108",
    "C110"
])

transactions = sc.parallelize([
    ("T001", "C101", 15000, "SUCCESS"),
    ("T002", "C102", 50000, "SUCCESS"),
    ("T003", "C101", 30000, "SUCCESS"),
    ("T004", "C103", 12000, "FAILED"),
    ("T005", "C103", 45000, "SUCCESS"),
    ("T006", "C104", 70000, "SUCCESS"),
    ("T007", "C105", 20000, "SUCCESS"),
    ("T008", "C105", 25000, "SUCCESS"),
    ("T009", "C108", 10000, "FAILED"),
    ("T010", "C108", 18000, "SUCCESS")
])

# REQUIREMENT:
#
# 1. Consider only successful transactions.
# 2. Calculate total spend per customer.
# 3. Keep customers whose total spend >= 40000.
# 4. From those customers, return only customers who are
#    present in the fraud watchlist.
#
# Expected:
#
# C101
# C103
# C105


# ============================================================
# QUESTION 12
# Find Customers New This Month
# Level: HARD
# ============================================================

july = sc.parallelize([
    ("T001", "C101", 1000),
    ("T002", "C102", 3000),
    ("T003", "C103", 2500),
    ("T004", "C101", 1800),
    ("T005", "C104", 5000)
])

august = sc.parallelize([
    ("T101", "C101", 2000),
    ("T102", "C103", 4000),
    ("T103", "C105", 3500),
    ("T104", "C106", 1000),
    ("T105", "C105", 2500),
    ("T106", "C107", 9000)
])

# REQUIREMENT:
#
# Find customers who appear in August but never appeared in July.
#
# A customer may have multiple transactions.
#
# Expected:
#
# C105
# C106
# C107


# ============================================================
# QUESTION 13
# Customers Active in Either Month But Not Both
# Level: HARD
# ============================================================

july = sc.parallelize([
    ("C101", 1000),
    ("C102", 2000),
    ("C103", 3000),
    ("C104", 4000),
    ("C101", 5000)
])

august = sc.parallelize([
    ("C101", 2000),
    ("C103", 3500),
    ("C105", 6000),
    ("C106", 8000),
    ("C105", 1200)
])

# REQUIREMENT:
#
# Find customers who were active in ONLY ONE of the two months.
#
# In other words:
#
# active in July but not August
#
# OR
#
# active in August but not July
#
# Expected:
#
# C102
# C104
# C105
# C106


# ============================================================
# QUESTION 14
# Failed Transaction Amount Per Customer
# Level: HARD
# ============================================================

transactions = sc.parallelize([
    ("T001", "C101", 12000, "SUCCESS"),
    ("T002", "C101", 5000,  "FAILED"),
    ("T003", "C102", 9000,  "FAILED"),
    ("T004", "C101", 7000,  "FAILED"),
    ("T005", "C103", 15000, "SUCCESS"),
    ("T006", "C102", 4000,  "FAILED"),
    ("T007", "C104", 20000, "FAILED"),
    ("T008", "C103", 8000,  "FAILED"),
    ("T009", "C104", 10000, "SUCCESS")
])

# REQUIREMENT:
#
# Consider only FAILED transactions.
#
# Calculate total failed transaction amount per customer.
#
# Then return only customers whose total FAILED amount
# is >= 10000.
#
# Expected:
#
# C101 -> 12000
# C102 -> 13000
# C104 -> 20000


# ============================================================
# QUESTION 15
# Detect Duplicate Transaction IDs
# Level: HARD
# ============================================================

transactions = sc.parallelize([
    ("T001", "C101", 1000),
    ("T002", "C102", 2000),
    ("T003", "C103", 3000),
    ("T001", "C101", 1000),
    ("T004", "C104", 4000),
    ("T002", "C102", 2000),
    ("T005", "C105", 5000),
    ("T001", "C101", 1000)
])

# REQUIREMENT:
#
# Find transaction IDs that occur MORE THAN ONCE.
#
# Output should also contain occurrence count.
#
# Expected:
#
# T001 -> 3
# T002 -> 2
#
# Do not solve this only by using Python collections.
# Use RDD transformations.


# ============================================================
# QUESTION 16
# Group S3 Files By Year-Month After Filtering
# Level: HARD
# ============================================================

paths = sc.parallelize([
    "s3://company/sales/year=2026/month=08/day=01/file1.parquet",
    "s3://company/sales/year=2026/month=08/day=02/file2.parquet",
    "s3://company/sales/year=2026/month=07/day=31/file3.parquet",
    "s3://company/logs/year=2026/month=08/day=01/log1.json",
    "s3://company/sales/year=2025/month=12/day=01/file4.parquet",
    "s3://company/logs/year=2025/month=12/day=02/log2.json",
    "s3://company/sales/year=2026/month=08/day=03/file5.parquet"
])

# REQUIREMENT:
#
# 1. Keep only paths belonging to /sales/.
# 2. Extract year and month.
# 3. Extract file name.
# 4. Group files by YYYY-MM.
#
# Expected conceptual output:
#
# 2026-08
#     file1.parquet
#     file2.parquet
#     file5.parquet
#
# 2026-07
#     file3.parquet
#
# 2025-12
#     file4.parquet


# ============================================================
# QUESTION 17
# Find Customers With Both Success and Failure
# Level: HARD
# ============================================================

transactions = sc.parallelize([
    ("T001", "C101", "SUCCESS"),
    ("T002", "C101", "FAILED"),
    ("T003", "C102", "SUCCESS"),
    ("T004", "C103", "FAILED"),
    ("T005", "C103", "FAILED"),
    ("T006", "C104", "SUCCESS"),
    ("T007", "C104", "FAILED"),
    ("T008", "C105", "SUCCESS"),
    ("T009", "C105", "SUCCESS"),
    ("T010", "C106", "FAILED")
])

# REQUIREMENT:
#
# Find customers who have at least:
#
# one SUCCESS transaction
#
# AND
#
# one FAILED transaction.
#
# Expected:
#
# C101
# C104


# ============================================================
# QUESTION 18
# Transaction Volume By Country
# Level: HARD
# ============================================================

transactions = sc.parallelize([
    ("T001", "C101", 12000, "INDIA",   "SUCCESS"),
    ("T002", "C102", 22000, "USA",     "SUCCESS"),
    ("T003", "C103", 5000,  "INDIA",   "FAILED"),
    ("T004", "C104", 32000, "USA",     "SUCCESS"),
    ("T005", "C105", 45000, "UK",      "SUCCESS"),
    ("T006", "C106", 8000,  "UK",      "SUCCESS"),
    ("T007", "C107", 51000, "USA",     "FAILED"),
    ("T008", "C108", 27000, "GERMANY", "SUCCESS"),
    ("T009", "C109", 18000, "INDIA",   "SUCCESS"),
    ("T010", "C110", 15000, "UK",      "SUCCESS")
])

# REQUIREMENT:
#
# Consider only SUCCESS transactions.
#
# For each country calculate:
#
# 1. Total successful transaction amount
# 2. Number of successful transactions
#
# Expected:
#
# INDIA   -> (30000, 2)
# USA     -> (54000, 2)
# UK      -> (68000, 3)
# GERMANY -> (27000, 1)


# ============================================================
# QUESTION 19
# High-Risk Customers Seen Across Two Systems
# Level: VERY HARD
# ============================================================

banking_system = sc.parallelize([
    ("C101", 120000),
    ("C102", 45000),
    ("C103", 95000),
    ("C104", 150000),
    ("C105", 70000),
    ("C106", 125000)
])

credit_card_system = sc.parallelize([
    ("C101", 40000),
    ("C103", 30000),
    ("C104", 60000),
    ("C105", 50000),
    ("C107", 140000),
    ("C108", 160000)
])

# REQUIREMENT:
#
# A customer is HIGH RISK if their total amount
# across BOTH systems is >= 150000.
#
# Customers can appear in one system or both.
#
# Calculate combined exposure per customer.
#
# Then return only HIGH-RISK customers.
#
# Expected:
#
# C101 -> 160000
# C104 -> 210000


# ============================================================
# QUESTION 20
# Complete Daily Reconciliation
# Level: VERY HARD / INTERVIEW LEVEL
# ============================================================

source_day1 = sc.parallelize([
    ("T001", "C101", 10000, "SUCCESS"),
    ("T002", "C102", 15000, "SUCCESS"),
    ("T003", "C103", 8000,  "FAILED"),
    ("T004", "C104", 22000, "SUCCESS")
])

source_day2 = sc.parallelize([
    ("T005", "C101", 12000, "SUCCESS"),
    ("T006", "C105", 30000, "SUCCESS"),
    ("T007", "C106", 9000,  "FAILED"),
    ("T008", "C102", 18000, "SUCCESS"),
    ("T009", "C107", 45000, "SUCCESS")
])

target = sc.parallelize([
    ("T001", "C101", 10000),
    ("T002", "C102", 15000),
    ("T004", "C104", 22000),
    ("T005", "C101", 12000),
    ("T008", "C102", 18000),
    ("T010", "C108", 50000)
])

# REQUIREMENT:
#
# PART 1:
# Combine source Day 1 and Day 2.
#
# PART 2:
# Keep only SUCCESSFUL source transactions.
#
# PART 3:
# Find:
#
#     MATCHED transaction IDs
#     MISSING transaction IDs
#     EXTRA transaction IDs
#
# PART 4:
# For the SUCCESSFUL source data,
# calculate total successful amount per customer.
#
# PART 5:
# Return customers whose successful source total
# is >= 30000.
#
#
# Expected reconciliation:
#
# MATCHED
# T001
# T002
# T004
# T005
# T008
#
# MISSING
# T006
# T009
#
# EXTRA
# T010
#
#
# Expected high-value customers:
#
# C102 -> 33000
# C105 -> 30000
# C107 -> 45000

### groupByKey()

- GroupByKey() works on a pair RDD (key,value)
- It groups all values having the same key 
- rdd.groupByKey(function)

- INPUT (key,value)

Purpose: Bring all value belonging to the same key togather 





In [ ]:
errors = [
    ("PAYMENT", "Database timeout"),
    ("ORDER", "API timeout"),
    ("PAYMENT", "Connection refused"),
    ("CUSTOMER", "Invalid token"),
    ("ORDER", "Out of memory"),
    ("PAYMENT", "Authentication failed")
]

rdd = sc.parallelize(errors, 3)

# Collect all error message belonging to each application 





In [ ]:
result=rdd.groupByKey()

for application,messages in result.collect():
    print("\nApplication",application)
    for message in messages:
        print(message)

GroupBy                                                     GroupByKey()

- Can create grouping key using a function                    key already exists 

- groupBy(function)                                           grouByKey() 

- Groups original records                                     Groups Values

- Wide                                                        Wide 

- Shuffle                                                     Shuffle 





In [ ]:
transactions = sc.parallelize([
    ("TXN001", "C101", "2026-08-19 09:10:00", "INDIA", "CARD", 1200, "SUCCESS"),
    ("TXN002", "C102", "2026-08-19 09:20:00", "INDIA", "UPI", 800, "SUCCESS"),
    ("TXN003", "C101", "2026-08-19 10:15:00", "INDIA", "CARD", 2500, "FAILED"),
    ("TXN004", "C103", "2026-08-19 10:45:00", "USA", "CARD", 3000, "SUCCESS"),
    ("TXN005", "C102", "2026-08-19 11:30:00", "INDIA", "UPI", 1500, "SUCCESS"),
    ("TXN006", "C104", "2026-08-19 12:10:00", "USA", "BANK", 5000, "SUCCESS"),
    ("TXN007", "C101", "2026-08-19 12:40:00", "INDIA", "CARD", 700, "SUCCESS"),
    ("TXN008", "C103", "2026-08-19 13:15:00", "USA", "UPI", 1100, "FAILED"),
    ("TXN009", "C105", "2026-08-19 14:10:00", "UK", "CARD", 4200, "SUCCESS"),
    ("TXN010", "C102", "2026-08-19 14:50:00", "INDIA", "BANK", 900, "FAILED"),
    ("TXN011", "C104", "2026-08-19 15:30:00", "USA", "CARD", 2200, "SUCCESS"),
    ("TXN012", "C105", "2026-08-19 16:20:00", "UK", "CARD", 1800, "SUCCESS")
], 4)

# Requirment 

    # - Group transection into:

    #     Morning   5:00-11:59
    #     Afternoon 12:00-16:59
    #     Evening   17:00-21-59
    #     NIght     22:00-4:59

    # GroupBy() - Original transection record 

# Requirment -2 

    # Collect transection details per customer 
    # c101-()
    # c102-()

    # groupByKey() - cutomer_id - all values belonging to customer 


# Requirment :3 

    # customer
    # total transection 
    # total amount 
    # sucessful trasection 
    # failed transection 


# c101-> transection-3 , amount -4400, sucess- 2 , failed- 1  

# reduceByKey() 

In [ ]:
def get_time_bucket(record):
    timestamp=record[2]
    #print(timestamp)
    #2026-08-19 09:10:00
    #hour=int(timestamp[11:13])
    date_part,time_part=timestamp.split()
    hour=int(time_part.split(":")[0])
    if 5<= hour <11:
        return "MORNING"
    elif 12<= hour <16:
        return "AFTERNOON"
    elif 17<= hour <21:
        return "EVENING"
    else:
        return "NIGHT"
    

In [ ]:
grouped=transactions.groupBy(get_time_bucket)
grouped.collect()
grouped.mapValues(list).collect()

In [ ]:
timestamp="2026-08-19 09:10:00"

date_part,time_part=timestamp.split()
print(date_part)
print(time_part)

In [ ]:
hour=int(time_part.split(":")[0])
print(hour)

## Why is reduceByKey() usually prefred over groupBykey() for aggregation?

- groupByKeyKey() groups all values for each key and sends the values through the shuffle so they can be collected  togather.

- reduceByKey() - Can perform map-side aggreagtion before shuffle, Therefore when the requirment is an assoctivate aggreagtion 

### aggregateByKey()

- Groups records by key and lets us perform aggregation in two stages :

synatx :

    rdd.aggregateByKey(zeroValue)(
        seqFunc,
        combFunc
    )

ZeroValue: Starting value for each key 

SeqFunc: How values are combined WITHIN a partition 

combfunc: How partital result are combined ACROSS partitions 





In [ ]:
transactions = sc.parallelize([
    ("C101", 100),
    ("C102", 200),
    ("C101", 300),
    ("C103", 400),
    ("C102", 500),
    ("C101", 600)
],3)


In [ ]:
def seq_func(result_so_far,value):
    return (
        result_so_far[0]+value,
        result_so_far[1]+1
    )

def comb_func(acc1,acc2):
    total=acc1[0]+acc2[0]
    count=acc1[1]+acc2[1]
    return (total,count)


result=transactions.aggregateByKey(
    (0,0),
    seq_func,
    comb_func
)

print(result.collect())

# seq_func(current_value,new_value)


In [ ]:
100
200
300
400

total=0
total=total+100 #100
total=total+200 # 300
total=total+300 # 600
total=total+400 # 1000

start->0
after 100-> 100
after 200 -> 300
after 300 -> 600
after 400 -> 1000



transection 

100
200
300 

We need 

total=600
count=3 


total=0 


(total,count)---> (0,0)

process 100:

previous result=(0,0)

new_value = 100 

new_result=(100,1)

process 200 :

previous result=(100,1)

new_value =200 

new_result=(300,2)

process 300


start
(0,0)
|||
| +100
|
(100,1)
|
| +200
|
(300,2)
|
|+300
|
(600,3). -> Final Output 


In [ ]:
sales=sc.parallelize([
    ("Apple",10),
    ("Banana",20),
    ("Apple",30),
    ("Banana",40),
    ("Apple",50)
])

In [ ]:
def add_values(total_so_far,new_value):
    return total_so_far+new_value

def combine_totals(total1,total2):
    return total1+total2


result=sales.aggregateByKey(
    0,
    add_values,
    combine_totals
)

print(result.collect())

# PySpark RDD — aggregateByKey() Practice Questions

## Levels

- Easy: Q1–Q15
- Medium: Q16–Q30
- Hard: Q31–Q40
- Real-World Data Engineering: Q41–Q50

---

# EASY LEVEL

## Q1. Total Sales Per Product

### Dataset

```python
sales = sc.parallelize([
    ("Apple", 10),
    ("Banana", 20),
    ("Apple", 30),
    ("Banana", 40),
    ("Apple", 50)
], 2)
```

### Requirement

Using `aggregateByKey()`:

- Group records logically by product.
- Calculate total sales for each product.

### Expected Output

```text
Apple  → 90
Banana → 60
```

---

## Q2. Total Salary Per Department

### Dataset

```python
employees = sc.parallelize([
    ("IT", 50000),
    ("HR", 40000),
    ("IT", 60000),
    ("HR", 45000),
    ("Finance", 70000)
], 2)
```

### Requirement

Calculate total salary paid by each department.

### Expected Output

```text
IT      → 110000
HR      → 85000
Finance → 70000
```

---

## Q3. Total Orders Per Customer

### Dataset

```python
orders = sc.parallelize([
    ("C101", 2),
    ("C102", 3),
    ("C101", 4),
    ("C103", 1),
    ("C102", 5)
], 2)
```

### Requirement

Calculate total number of ordered items for each customer.

### Expected Output

```text
C101 → 6
C102 → 8
C103 → 1
```

---

## Q4. Total Website Visits

### Dataset

```python
visits = sc.parallelize([
    ("google.com", 10),
    ("amazon.com", 20),
    ("google.com", 15),
    ("amazon.com", 30),
    ("openai.com", 5)
], 2)
```

### Requirement

Calculate total visits for each website.

### Expected Output

```text
google.com → 25
amazon.com → 50
openai.com → 5
```

---

## Q5. Total Marks Per Student

### Dataset

```python
marks = sc.parallelize([
    ("Rahul", 70),
    ("Amit", 80),
    ("Rahul", 90),
    ("Amit", 60),
    ("Neha", 85)
], 2)
```

### Requirement

Calculate total marks for each student.

### Expected Output

```text
Rahul → 160
Amit  → 140
Neha  → 85
```

---

## Q6. Maximum Transaction Per Customer

### Dataset

```python
transactions = sc.parallelize([
    ("C101", 100),
    ("C102", 500),
    ("C101", 800),
    ("C102", 300),
    ("C101", 250)
], 2)
```

### Requirement

Find the maximum transaction amount for each customer using `aggregateByKey()`.

### Expected Output

```text
C101 → 800
C102 → 500
```

---

## Q7. Minimum Transaction Per Customer

### Dataset

```python
transactions = sc.parallelize([
    ("C101", 100),
    ("C102", 500),
    ("C101", 800),
    ("C102", 300),
    ("C101", 250)
], 2)
```

### Requirement

Find the minimum transaction amount for each customer.

### Expected Output

```text
C101 → 100
C102 → 300
```

---

## Q8. Count Transactions Per Customer

### Dataset

```python
transactions = sc.parallelize([
    ("C101", 100),
    ("C102", 500),
    ("C101", 800),
    ("C102", 300),
    ("C101", 250)
], 2)
```

### Requirement

Count how many transactions each customer made.

Do NOT calculate the transaction amount.

### Expected Output

```text
C101 → 3
C102 → 2
```

---

## Q9. Total Units Sold Per Product

### Dataset

```python
products = sc.parallelize([
    ("Laptop", 2),
    ("Mobile", 5),
    ("Laptop", 3),
    ("Tablet", 4),
    ("Mobile", 2)
], 2)
```

### Requirement

Calculate total units sold for every product.

### Expected Output

```text
Laptop → 5
Mobile → 7
Tablet → 4
```

---

## Q10. Total Error Count Per Error Type

### Dataset

```python
errors = sc.parallelize([
    ("404", 10),
    ("500", 5),
    ("404", 20),
    ("403", 7),
    ("500", 8)
], 2)
```

### Requirement

Calculate total occurrences for every error type.

### Expected Output

```text
404 → 30
500 → 13
403 → 7
```

---

## Q11. Maximum Temperature Per City

### Dataset

```python
temperatures = sc.parallelize([
    ("Delhi", 35),
    ("Mumbai", 32),
    ("Delhi", 41),
    ("Mumbai", 36),
    ("Delhi", 38)
], 2)
```

### Requirement

Find the highest recorded temperature for every city.

### Expected Output

```text
Delhi  → 41
Mumbai → 36
```

---

## Q12. Minimum Response Time Per API

### Dataset

```python
api_logs = sc.parallelize([
    ("login", 120),
    ("payment", 300),
    ("login", 90),
    ("payment", 250),
    ("search", 80)
], 2)
```

### Requirement

Find the minimum response time for every API.

### Expected Output

```text
login   → 90
payment → 250
search  → 80
```

---

## Q13. Total File Size Per File Type

### Dataset

```python
files = sc.parallelize([
    ("csv", 100),
    ("json", 200),
    ("csv", 150),
    ("parquet", 500),
    ("json", 300)
], 2)
```

### Requirement

Calculate total file size for each file type.

### Expected Output

```text
csv     → 250
json    → 500
parquet → 500
```

---

## Q14. Total Revenue Per Store

### Dataset

```python
revenue = sc.parallelize([
    ("Store-A", 1000),
    ("Store-B", 1500),
    ("Store-A", 2000),
    ("Store-C", 800),
    ("Store-B", 500)
], 2)
```

### Requirement

Calculate total revenue for every store.

### Expected Output

```text
Store-A → 3000
Store-B → 2000
Store-C → 800
```

---

## Q15. Total Data Processed Per Job

### Dataset

```python
jobs = sc.parallelize([
    ("job1", 100),
    ("job2", 200),
    ("job1", 300),
    ("job3", 150),
    ("job2", 400)
], 2)
```

### Requirement

Calculate total MB processed by every job.

### Expected Output

```text
job1 → 400
job2 → 600
job3 → 150
```

---

# MEDIUM LEVEL

Now start using different input and output structures.

---

## Q16. Total Amount AND Transaction Count

### Dataset

```python
transactions = sc.parallelize([
    ("C101", 100),
    ("C102", 200),
    ("C101", 300),
    ("C102", 400),
    ("C101", 500)
], 3)
```

### Requirement

For every customer calculate:

```text
(total_amount, transaction_count)
```

### Expected Output

```text
C101 → (900, 3)
C102 → (600, 2)
```

Hint:

```text
zeroValue = (0, 0)
```

---

## Q17. Average Transaction Amount

### Dataset

```python
transactions = sc.parallelize([
    ("C101", 100),
    ("C102", 200),
    ("C101", 300),
    ("C102", 400),
    ("C101", 500)
], 3)
```

### Requirement

Using `aggregateByKey()`:

1. Calculate `(total, count)`.
2. Then calculate average using `mapValues()`.

### Expected Output

```text
C101 → 300.0
C102 → 300.0
```

---

## Q18. Average Salary Per Department

### Dataset

```python
employees = sc.parallelize([
    ("IT", 50000),
    ("HR", 40000),
    ("IT", 70000),
    ("HR", 60000),
    ("IT", 90000)
], 3)
```

### Requirement

Calculate average salary per department.

### Expected Output

```text
IT → 70000
HR → 50000
```

---

## Q19. Sum AND Count Website Response Times

### Dataset

```python
logs = sc.parallelize([
    ("login", 100),
    ("search", 200),
    ("login", 150),
    ("search", 300),
    ("login", 250)
], 3)
```

### Requirement

Calculate:

```text
API → (total_response_time, request_count)
```

### Expected Output

```text
login  → (500, 3)
search → (500, 2)
```

---

## Q20. Minimum AND Maximum Transaction

### Dataset

```python
transactions = sc.parallelize([
    ("C101", 100),
    ("C102", 500),
    ("C101", 900),
    ("C102", 200),
    ("C101", 400)
], 3)
```

### Requirement

For every customer calculate:

```text
(min_transaction, max_transaction)
```

### Expected Output

```text
C101 → (100, 900)
C102 → (200, 500)
```

---

## Q21. Total, Count AND Average

### Dataset

```python
sales = sc.parallelize([
    ("Laptop", 1000),
    ("Mobile", 500),
    ("Laptop", 2000),
    ("Mobile", 1500),
    ("Laptop", 3000)
], 3)
```

### Requirement

Calculate:

```text
Product → (total, count, average)
```

### Expected Output

```text
Laptop → (6000, 3, 2000)
Mobile → (2000, 2, 1000)
```

---

## Q22. Collect Values Into a List

### Dataset

```python
orders = sc.parallelize([
    ("C101", "O1"),
    ("C102", "O2"),
    ("C101", "O3"),
    ("C102", "O4"),
    ("C101", "O5")
], 3)
```

### Requirement

Using `aggregateByKey()`, collect order IDs belonging to each customer.

### Expected Output

```text
C101 → ["O1", "O3", "O5"]
C102 → ["O2", "O4"]
```

Do NOT use `groupByKey()`.

---

## Q23. Collect Unique Cities Per Customer

### Dataset

```python
visits = sc.parallelize([
    ("C101", "Delhi"),
    ("C101", "Mumbai"),
    ("C102", "Delhi"),
    ("C101", "Delhi"),
    ("C102", "Pune")
], 3)
```

### Requirement

Return unique cities visited by each customer.

### Expected Output

```text
C101 → {"Delhi", "Mumbai"}
C102 → {"Delhi", "Pune"}
```

---

## Q24. Number of Unique Cities

Use the same dataset from Q23.

### Requirement

Calculate the number of unique cities visited by each customer.

### Expected Output

```text
C101 → 2
C102 → 2
```

---

## Q25. Maximum Order AND Order Count

### Dataset

```python
orders = sc.parallelize([
    ("C101", 500),
    ("C102", 200),
    ("C101", 900),
    ("C102", 700),
    ("C101", 300)
], 3)
```

### Requirement

Calculate:

```text
Customer → (maximum_order, number_of_orders)
```

### Expected Output

```text
C101 → (900, 3)
C102 → (700, 2)
```

---

## Q26. Success and Failure Counts

### Dataset

```python
jobs = sc.parallelize([
    ("pipeline1", "SUCCESS"),
    ("pipeline1", "FAILED"),
    ("pipeline2", "SUCCESS"),
    ("pipeline1", "SUCCESS"),
    ("pipeline2", "FAILED"),
    ("pipeline2", "FAILED")
], 3)
```

### Requirement

For every pipeline calculate:

```text
(success_count, failure_count)
```

### Expected Output

```text
pipeline1 → (2, 1)
pipeline2 → (1, 2)
```

---

## Q27. Positive and Negative Transaction Counts

### Dataset

```python
transactions = sc.parallelize([
    ("C101", 100),
    ("C101", -50),
    ("C102", 200),
    ("C101", -20),
    ("C102", -100)
], 3)
```

### Requirement

For every customer calculate:

```text
(positive_transaction_count, negative_transaction_count)
```

### Expected Output

```text
C101 → (1, 2)
C102 → (1, 1)
```

---

## Q28. Total Credit and Debit Amount

### Dataset

```python
transactions = sc.parallelize([
    ("C101", ("CREDIT", 1000)),
    ("C101", ("DEBIT", 300)),
    ("C102", ("CREDIT", 500)),
    ("C101", ("CREDIT", 700)),
    ("C102", ("DEBIT", 200))
], 3)
```

### Requirement

Calculate:

```text
Customer → (total_credit, total_debit)
```

### Expected Output

```text
C101 → (1700, 300)
C102 → (500, 200)
```

---

## Q29. Count HTTP Status Categories

### Dataset

```python
logs = sc.parallelize([
    ("api1", 200),
    ("api1", 500),
    ("api2", 200),
    ("api1", 404),
    ("api2", 503),
    ("api2", 201)
], 3)
```

### Requirement

For every API calculate:

```text
(success_count, error_count)
```

Treat:

```text
200–299 → SUCCESS
Everything else → ERROR
```

### Expected Output

```text
api1 → (1, 2)
api2 → (2, 1)
```

---

## Q30. Earliest and Latest Timestamp

### Dataset

```python
events = sc.parallelize([
    ("C101", "2026-08-20 10:00:00"),
    ("C102", "2026-08-20 09:00:00"),
    ("C101", "2026-08-20 15:00:00"),
    ("C102", "2026-08-20 18:00:00"),
    ("C101", "2026-08-20 08:00:00")
], 3)
```

### Requirement

Find:

```text
Customer → (earliest_timestamp, latest_timestamp)
```

### Expected Output

```text
C101 → ("2026-08-20 08:00:00", "2026-08-20 15:00:00")
C102 → ("2026-08-20 09:00:00", "2026-08-20 18:00:00")
```

---

# HARD LEVEL

## Q31. Transaction Statistics

### Dataset

```python
transactions = sc.parallelize([
    ("C101", 100),
    ("C102", 500),
    ("C101", 300),
    ("C101", 900),
    ("C102", 200),
    ("C102", 700)
], 4)
```

### Requirement

For each customer calculate:

```text
(total, count, minimum, maximum)
```

### Expected Output

```text
C101 → (1300, 3, 100, 900)
C102 → (1400, 3, 200, 700)
```

---

## Q32. Complete Transaction Statistics

Use Q31 dataset.

### Requirement

Calculate:

```text
Customer →
(
    total,
    count,
    minimum,
    maximum,
    average
)
```

### Expected Output

```text
C101 → (1300, 3, 100, 900, 433.33)
C102 → (1400, 3, 200, 700, 466.67)
```

---

## Q33. Error Statistics Per Application

### Dataset

```python
logs = sc.parallelize([
    ("payment", ("ERROR", 500)),
    ("payment", ("SUCCESS", 120)),
    ("login", ("ERROR", 300)),
    ("payment", ("ERROR", 700)),
    ("login", ("SUCCESS", 100)),
    ("login", ("ERROR", 400))
], 4)
```

### Requirement

For every application calculate:

```text
(
    success_count,
    error_count,
    total_error_response_time
)
```

### Expected Output

```text
payment → (1, 2, 1200)
login   → (1, 2, 700)
```

---

## Q34. Purchase Statistics Per Customer

### Dataset

```python
purchases = sc.parallelize([
    ("C101", ("Laptop", 1000)),
    ("C101", ("Mobile", 500)),
    ("C102", ("Tablet", 700)),
    ("C101", ("Laptop", 1200)),
    ("C102", ("Mobile", 800))
], 4)
```

### Requirement

For each customer calculate:

```text
(
    total_spent,
    purchase_count,
    unique_products
)
```

### Expected Output

```text
C101 → (2700, 3, {"Laptop", "Mobile"})
C102 → (1500, 2, {"Tablet", "Mobile"})
```

---

## Q35. Average Response Time of Successful Requests Only

### Dataset

```python
logs = sc.parallelize([
    ("login", (200, 100)),
    ("login", (500, 900)),
    ("login", (200, 200)),
    ("payment", (200, 400)),
    ("payment", (503, 1000)),
    ("payment", (201, 600))
], 4)
```

Each value contains:

```text
(status_code, response_time)
```

### Requirement

Ignore failed requests.

Calculate average response time only for HTTP `2xx` requests.

### Expected Output

```text
login   → 150
payment → 500
```

---

## Q36. High-Value vs Normal Transactions

### Dataset

```python
transactions = sc.parallelize([
    ("C101", 100),
    ("C101", 1500),
    ("C102", 700),
    ("C101", 2000),
    ("C102", 1200),
    ("C102", 300)
], 4)
```

### Requirement

Transaction >= 1000 is `HIGH_VALUE`.

Calculate:

```text
Customer →
(
    high_value_count,
    normal_count,
    high_value_total
)
```

### Expected Output

```text
C101 → (2, 1, 3500)
C102 → (1, 2, 1200)
```

---

## Q37. File Processing Statistics

### Dataset

```python
files = sc.parallelize([
    ("pipeline1", ("file1.csv", 100)),
    ("pipeline1", ("file2.csv", 300)),
    ("pipeline2", ("file3.csv", 500)),
    ("pipeline1", ("file4.csv", 200)),
    ("pipeline2", ("file5.csv", 700))
], 4)
```

### Requirement

For every pipeline calculate:

```text
(
    number_of_files,
    total_size,
    maximum_file_size
)
```

### Expected Output

```text
pipeline1 → (3, 600, 300)
pipeline2 → (2, 1200, 700)
```

---

## Q38. Customer Activity Summary

### Dataset

```python
events = sc.parallelize([
    ("C101", ("LOGIN", "2026-08-20 10:00:00")),
    ("C101", ("PURCHASE", "2026-08-20 11:00:00")),
    ("C102", ("LOGIN", "2026-08-20 09:00:00")),
    ("C101", ("LOGOUT", "2026-08-20 15:00:00")),
    ("C102", ("PURCHASE", "2026-08-20 12:00:00"))
], 4)
```

### Requirement

Calculate:

```text
Customer →
(
    event_count,
    earliest_event_time,
    latest_event_time
)
```

### Expected Output

```text
C101 → (3, "2026-08-20 10:00:00", "2026-08-20 15:00:00")
C102 → (2, "2026-08-20 09:00:00", "2026-08-20 12:00:00")
```

---

## Q39. Data Quality Statistics

### Dataset

```python
records = sc.parallelize([
    ("customer", "VALID"),
    ("customer", "INVALID"),
    ("orders", "VALID"),
    ("customer", "VALID"),
    ("orders", "INVALID"),
    ("orders", "INVALID"),
    ("payments", "VALID")
], 4)
```

### Requirement

For every dataset calculate:

```text
(
    total_records,
    valid_records,
    invalid_records
)
```

### Expected Output

```text
customer → (3, 2, 1)
orders   → (3, 1, 2)
payments → (1, 1, 0)
```

---

## Q40. Sales by Region With Multiple Metrics

### Dataset

```python
sales = sc.parallelize([
    ("North", ("Laptop", 1000)),
    ("South", ("Mobile", 500)),
    ("North", ("Mobile", 700)),
    ("North", ("Laptop", 1500)),
    ("South", ("Tablet", 900)),
    ("South", ("Mobile", 600))
], 4)
```

### Requirement

For every region calculate:

```text
(
    total_revenue,
    transaction_count,
    unique_products,
    maximum_sale
)
```

### Expected Output

```text
North → (3200, 3, {"Laptop", "Mobile"}, 1500)
South → (2000, 3, {"Mobile", "Tablet"}, 900)
```

---

# 🚀 REAL-WORLD DATA ENGINEERING LEVEL

## Q41. API Monitoring Aggregation

### Dataset

```python
api_logs = sc.parallelize([
    ("payment-api", (200, 120)),
    ("payment-api", (500, 900)),
    ("login-api", (200, 80)),
    ("payment-api", (200, 180)),
    ("login-api", (503, 700)),
    ("login-api", (200, 100))
], 4)
```

Value:

```text
(status_code, response_time_ms)
```

### Requirement

Calculate per API:

```text
(
    total_requests,
    successful_requests,
    failed_requests,
    total_response_time
)
```

### Expected Output

```text
payment-api → (3, 2, 1, 1200)
login-api   → (3, 2, 1, 880)
```

---

## Q42. ETL Pipeline Execution Statistics

### Dataset

```python
jobs = sc.parallelize([
    ("sales_etl", ("SUCCESS", 120)),
    ("sales_etl", ("FAILED", 40)),
    ("customer_etl", ("SUCCESS", 200)),
    ("sales_etl", ("SUCCESS", 100)),
    ("customer_etl", ("SUCCESS", 180)),
    ("customer_etl", ("FAILED", 50))
], 4)
```

Value:

```text
(status, execution_seconds)
```

### Requirement

Calculate:

```text
Pipeline →
(
    run_count,
    success_count,
    failure_count,
    total_execution_time
)
```

### Expected Output

```text
sales_etl    → (3, 2, 1, 260)
customer_etl → (3, 2, 1, 430)
```

---

## Q43. S3 File Ingestion Statistics

### Dataset

```python
files = sc.parallelize([
    ("sales", ("file1.csv", 100)),
    ("customer", ("file2.csv", 300)),
    ("sales", ("file3.csv", 500)),
    ("sales", ("file4.csv", 200)),
    ("customer", ("file5.csv", 700))
], 4)
```

Value:

```text
(file_name, size_mb)
```

### Requirement

For every dataset calculate:

```text
(
    file_count,
    total_size_mb,
    largest_file_mb
)
```

### Expected Output

```text
sales    → (3, 800, 500)
customer → (2, 1000, 700)
```

---

## Q44. Banking Transaction Summary

### Dataset

```python
transactions = sc.parallelize([
    ("C101", ("CREDIT", 1000)),
    ("C101", ("DEBIT", 300)),
    ("C102", ("CREDIT", 2000)),
    ("C101", ("DEBIT", 500)),
    ("C102", ("DEBIT", 700)),
    ("C102", ("CREDIT", 500))
], 4)
```

### Requirement

Calculate:

```text
Customer →
(
    credit_count,
    debit_count,
    total_credit,
    total_debit,
    balance_change
)
```

Where:

```text
balance_change = total_credit - total_debit
```

### Expected Output

```text
C101 → (1, 2, 1000, 800, 200)
C102 → (2, 1, 2500, 700, 1800)
```

---

## Q45. Failed Job Error Collection

### Dataset

```python
jobs = sc.parallelize([
    ("sales_etl", ("SUCCESS", None)),
    ("sales_etl", ("FAILED", "S3_ACCESS_ERROR")),
    ("customer_etl", ("FAILED", "TIMEOUT")),
    ("sales_etl", ("FAILED", "MEMORY_ERROR")),
    ("customer_etl", ("SUCCESS", None))
], 4)
```

### Requirement

For every pipeline return:

```text
(
    failure_count,
    unique_error_types
)
```

### Expected Output

```text
sales_etl    → (2, {"S3_ACCESS_ERROR", "MEMORY_ERROR"})
customer_etl → (1, {"TIMEOUT"})
```

---

## Q46. Daily Sales Aggregation

### Dataset

```python
sales = sc.parallelize([
    (("2026-08-20", "Laptop"), 1000),
    (("2026-08-20", "Mobile"), 500),
    (("2026-08-20", "Laptop"), 1500),
    (("2026-08-21", "Laptop"), 800),
    (("2026-08-21", "Mobile"), 700),
    (("2026-08-21", "Mobile"), 900)
], 4)
```

### Requirement

Using the composite key:

```text
(date, product)
```

calculate:

```text
(total_sales, transaction_count)
```

### Expected Output

```text
("2026-08-20", "Laptop") → (2500, 2)
("2026-08-20", "Mobile") → (500, 1)
("2026-08-21", "Laptop") → (800, 1)
("2026-08-21", "Mobile") → (1600, 2)
```

---

## Q47. Customer Fraud Monitoring

### Dataset

```python
transactions = sc.parallelize([
    ("C101", ("Delhi", 500)),
    ("C101", ("Mumbai", 15000)),
    ("C102", ("Delhi", 800)),
    ("C101", ("Pune", 20000)),
    ("C102", ("Delhi", 12000)),
    ("C102", ("Mumbai", 400))
], 4)
```

### Requirement

A transaction greater than `10000` is considered high-value.

For each customer calculate:

```text
(
    total_transactions,
    high_value_transactions,
    total_amount,
    unique_cities
)
```

### Expected Output

```text
C101 → (3, 2, 35500, {"Delhi", "Mumbai", "Pune"})
C102 → (3, 1, 13200, {"Delhi", "Mumbai"})
```

---

## Q48. Spark Job Partition Processing Summary

### Dataset

```python
processing = sc.parallelize([
    ("job1", ("partition-0", 1000, "SUCCESS")),
    ("job1", ("partition-1", 1200, "SUCCESS")),
    ("job2", ("partition-0", 500, "FAILED")),
    ("job1", ("partition-2", 800, "FAILED")),
    ("job2", ("partition-1", 700, "SUCCESS"))
], 4)
```

Value:

```text
(partition_id, records_processed, status)
```

### Requirement

Calculate:

```text
Job →
(
    partition_count,
    total_records_processed,
    successful_partitions,
    failed_partitions
)
```

### Expected Output

```text
job1 → (3, 3000, 2, 1)
job2 → (2, 1200, 1, 1)
```

---

## Q49. Data Quality Report

### Dataset

```python
quality = sc.parallelize([
    ("customers", ("VALID", 100)),
    ("customers", ("NULL_EMAIL", 10)),
    ("customers", ("NULL_PHONE", 5)),
    ("orders", ("VALID", 200)),
    ("orders", ("NULL_CUSTOMER_ID", 20)),
    ("customers", ("VALID", 50))
], 4)
```

Value:

```text
(record_status, record_count)
```

### Requirement

For every dataset calculate:

```text
(
    total_records,
    valid_records,
    invalid_records,
    unique_error_types
)
```

Do not include `VALID` in `unique_error_types`.

### Expected Output

```text
customers →
(
    165,
    150,
    15,
    {"NULL_EMAIL", "NULL_PHONE"}
)

orders →
(
    220,
    200,
    20,
    {"NULL_CUSTOMER_ID"}
)
```

---

# Q50. FINAL CHALLENGE — Production Transaction Aggregation

### Dataset

```python
transactions = sc.parallelize([
    ("C101", ("CREDIT", 1000, "Delhi", "SUCCESS")),
    ("C101", ("DEBIT", 300, "Delhi", "SUCCESS")),
    ("C102", ("CREDIT", 2000, "Mumbai", "SUCCESS")),
    ("C101", ("CREDIT", 15000, "Mumbai", "SUCCESS")),
    ("C102", ("DEBIT", 500, "Delhi", "FAILED")),
    ("C101", ("DEBIT", 700, "Pune", "FAILED")),
    ("C102", ("CREDIT", 12000, "Mumbai", "SUCCESS")),
    ("C102", ("DEBIT", 1000, "Pune", "SUCCESS"))
], 4)
```

Each value contains:

```text
(
    transaction_type,
    amount,
    city,
    status
)
```

### Requirement

Using `aggregateByKey()`, create a customer-level transaction summary.

For every customer calculate:

```text
(
    total_transactions,
    successful_transactions,
    failed_transactions,
    total_credit,
    total_debit,
    high_value_transactions,
    unique_cities
)
```

A transaction is `HIGH_VALUE` when:

```text
amount >= 10000
```

### Expected Output

```text
C101 →
(
    4,
    3,
    1,
    16000,
    1000,
    1,
    {"Delhi", "Mumbai", "Pune"}
)

C102 →
(
    4,
    3,
    1,
    14000,
    1500,
    1,
    {"Mumbai", "Delhi", "Pune"}
)
```

---

# aggregateByKey() Template

Try solving each question using this structure:

```python
def seq_func(result_so_far, new_value):

    # Process one original value
    # and update result_so_far

    return updated_result


def comb_func(result1, result2):

    # Combine partial results
    # produced by different partitions

    return combined_result


result = rdd.aggregateByKey(
    zero_value,
    seq_func,
    comb_func
)

print(result.collect())
```

---

# For Your Understanding 

```text
Original Records
       |
       v
+-------------------+
|    Partition 0    |
|                   |
| seq_func()        |
| seq_func()        |
| seq_func()        |
+-------------------+
       |
       | Partial Result
       |
       |                 +-------------------+
       |                 |    Partition 1    |
       |                 |                   |
       |                 | seq_func()        |
       |                 | seq_func()        |
       |                 +-------------------+
       |                           |
       |                           | Partial Result
       |                           |
       +------------+--------------+
                    |
                 SHUFFLE
                    |
                    v
              comb_func()
                    |
                    v
              Final Result
```

# Most Important Rule

```text
seq_func:

result_so_far + ONE ORIGINAL VALUE
              ↓
       updated result


comb_func:

PARTIAL RESULT + PARTIAL RESULT
              ↓
        final/combined result
```

In [ ]:
# mapValues() - It applies a function only to the value 
# Key remains unchanges 
# Narrow Transformation 


#- (key,value)
#       |
#    Changed


In [2]:
employee=sc.parallelize([
    ("E101",500000),
    ("E102",600000),
    ("E103",700000),

])

# Requirement 
# Give every employee a 5000 salary increment 

def add_increment(salary):
    return salary+5000

result=employee.mapValues(add_increment)
print(result.collect())

[('E101', 505000), ('E102', 605000), ('E103', 705000)]


In [6]:
def add_increment(record):
    employee_id=record[0]
    salary=record[1]
    return (employee_id,salary+5000)

result=employee.map(add_increment)

In [7]:
result.collect()

[('E101', 505000), ('E102', 605000), ('E103', 705000)]

In [8]:
transections=sc.parallelize([
    ("c101",[1000,2000,3000]),
    ("c102",[3000,1500]),
    ("c103",[700,800,1000])
])

#Requirment 

# Calculate total transection amount for every customer 

def calculate_total(amount):
    return sum(amount)

result=transections.mapValues(calculate_total)
print(result.collect())

[('c101', 6000), ('c102', 4500), ('c103', 2500)]


In [9]:
# flatMapValues() - 

# MapValues() - One input value -> One output value 
# FlatMapvalues() - One Input Value -> Zero,one or many output values 

purchase=sc.parallelize([
    ("c101",["laptop","Mouse"]),
    ("c102",["Phone","charger","Case"]),
    ("c103",["Keyboard"])
])

# Requirment 
#Create one (customer,product) record for every product 

result=purchase.flatMapValues(lambda product:product)
print(result.collect())

[('c101', 'laptop'), ('c101', 'Mouse'), ('c102', 'Phone'), ('c102', 'charger'), ('c102', 'Case'), ('c103', 'Keyboard')]


In [10]:
def get_product(products):
    return products
result=purchase.flatMapValues(get_product)
print(result.collect())


[('c101', 'laptop'), ('c101', 'Mouse'), ('c102', 'Phone'), ('c102', 'charger'), ('c102', 'Case'), ('c103', 'Keyboard')]


In [13]:
# keys() - Extract only keys from a pair RDD and removes the values 
    # - keys() does not remove the duplicate 
#  values() - Extracts only the values from a pair RDD and removes the keys 

employee=sc.parallelize([
    ("E101",500000),
    ("E102",600000),
    ("E103",700000),

])

purchase=sc.parallelize([
    ("c101",["laptop","Mouse"]),
    ("c102",["Phone","charger","Case"]),
    ("c103",["Keyboard"])
])
# (key,value)

# result=employee.keys()
# print(result.collect())

# result=employee.values()
# print(result.collect())
result=purchase.values()
print(result.collect())

[['laptop', 'Mouse'], ['Phone', 'charger', 'Case'], ['Keyboard']]


In [ ]:
transactions = sc.parallelize([
    ("C101", {"txn_id": "T001", "amount": 1200, "status": "SUCCESS", "country": "IN"}),
    ("C102", {"txn_id": "T002", "amount": 800,  "status": "FAILED",  "country": "IN"}),
    ("C101", {"txn_id": "T003", "amount": 2500, "status": "SUCCESS", "country": "US"}),
    ("C103", {"txn_id": "T004", "amount": 4000, "status": "SUCCESS", "country": "IN"}),
    ("C104", {"txn_id": "T005", "amount": 700,  "status": "FAILED",  "country": "UK"}),
    ("C102", {"txn_id": "T006", "amount": 1500, "status": "SUCCESS", "country": "IN"}),
    ("C105", {"txn_id": "T007", "amount": 3000, "status": "SUCCESS", "country": "US"}),
    ("C103", {"txn_id": "T008", "amount": 1000, "status": "FAILED",  "country": "IN"}),
    ("C101", {"txn_id": "T009", "amount": 1800, "status": "SUCCESS", "country": "IN"}),
    ("C106", {"txn_id": "T010", "amount": 5000, "status": "SUCCESS", "country": "UK"})
], 3)

# Requirment 1:

# Find unique customers who had sucessful transections 


def successful(record):
    txn=record[1]
    return txn["status"]=="SUCCESS"

successful_customers=(
    transactions
    .filter(successful)
    .keys()
    .distinct()
)

print(successful_customers.collect())


# Requirment 2

# Extact only successful transcrtion amounts 

def get_amount(txn):
    return txn["amount"]

result=(
    transactions
    .filter(successful)
    .values()
    .map(get_amount)
)

print(result.collect())


# Requirment 3 :

# - Find all unique customers who made at least one sucessfull tranection above 2000




['C102', 'C103', 'C101', 'C105', 'C106']
[1200, 2500, 4000, 1500, 3000, 1800, 5000]
